# Attention Residuals — PyTorch Implementation

> *Kimi Team (Mar 2026)* — [github.com/MoonshotAI/Attention-Residuals](https://github.com/MoonshotAI/Attention-Residuals)

Residual connections with PreNorm are standard in modern LLMs, yet they accumulate all layer outputs with fixed unit weights. This uniform aggregation causes uncontrolled hidden-state growth with depth, progressively diluting each layer's contribution. **Attention Residuals (AttnRes)** replaces this fixed accumulation with **softmax attention over preceding layer outputs**, allowing each layer to selectively aggregate earlier representations with learned, input-dependent weights.

This notebook implements two variants:
1. **Full Attention Residuals** — each layer attends over all previous layer outputs
2. **Block Attention Residuals** — layers are grouped into blocks, attending over block-level representations

![Attention Residuals Overview](attention-residuals.png)

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch import Tensor
import math

## Background: Standard Residual Connections

In standard Transformer models, each layer updates the hidden state via a simple additive residual:

$$\mathbf{h}_l = \mathbf{h}_{l-1} + f_{l-1}(\mathbf{h}_{l-1})$$

Expanding the recurrence, the hidden state at layer $l$ is the sum of the embedding and all preceding layer outputs:

$$\mathbf{h}_l = \mathbf{h}_1 + \sum_{i=1}^{l-1} f_i(\mathbf{h}_i)$$

**Limitations:** Every layer's contribution is weighted uniformly (coefficient = 1), giving no mechanism to adapt mixing across depth. This causes:
- Hidden-state growth with depth, diluting individual layer contributions
- No selective access to earlier representations
- Deeper layers must learn increasingly large outputs to influence the accumulated residual

In [6]:
class RMSNorm(nn.Module):
    """Root Mean Square Layer Normalization (Zhang & Sennrich, 2019)."""

    def __init__(self, dim: int, eps: float = 1e-8):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x: Tensor) -> Tensor:
        rms = x.norm(2, dim=-1, keepdim=True) / math.sqrt(x.shape[-1])
        return self.weight * x / (rms + self.eps)


class FeedForward(nn.Module):
    """Simple SwiGLU-style FFN used as the MLP sub-layer."""

    def __init__(self, dim: int, hidden_dim: int | None = None):
        super().__init__()
        hidden_dim = hidden_dim or 4 * dim
        self.w1 = nn.Linear(dim, hidden_dim, bias=False)
        self.w2 = nn.Linear(hidden_dim, dim, bias=False)
        self.w3 = nn.Linear(dim, hidden_dim, bias=False)

    def forward(self, x: Tensor) -> Tensor:
        return self.w2(F.silu(self.w1(x)) * self.w3(x))


class CausalSelfAttention(nn.Module):
    """Multi-head causal self-attention (simplified, no KV cache)."""

    def __init__(self, dim: int, n_heads: int):
        super().__init__()
        assert dim % n_heads == 0
        self.n_heads = n_heads
        self.head_dim = dim // n_heads
        self.qkv = nn.Linear(dim, 3 * dim, bias=False)
        self.out_proj = nn.Linear(dim, dim, bias=False)

    def forward(self, x: Tensor) -> Tensor:
        B, T, D = x.shape
        qkv = self.qkv(x).reshape(B, T, 3, self.n_heads, self.head_dim)
        q, k, v = qkv.unbind(dim=2)                        # each: [B, T, n_heads, head_dim]
        q, k, v = (t.transpose(1, 2) for t in (q, k, v))   # [B, n_heads, T, head_dim]

        # scaled dot-product attention with causal mask
        attn = F.scaled_dot_product_attention(q, k, v, is_causal=True)
        attn = attn.transpose(1, 2).reshape(B, T, D)
        return self.out_proj(attn)


print("Building blocks ready")

Building blocks ready


## 1. Full Attention Residuals

> *"Like RNNs over time, residual connections compress all prior information into a single state over depth. We propose the same methodology for depth."*

Instead of fixed-weight accumulation, each layer **attends over all preceding layer outputs** with learned, input-dependent weights:

$$\mathbf{h}_l = \sum_{i=0}^{l-1} \alpha_{i \to l} \cdot \mathbf{v}_i$$

where the attention weights are computed via softmax over depth:

$$\alpha_{i \to l} = \frac{\phi(\mathbf{q}_l,\, \mathbf{k}_i)}{\sum_{j=0}^{l-1} \phi(\mathbf{q}_l,\, \mathbf{k}_j)}$$

with kernel $\phi(\mathbf{q}, \mathbf{k}) = \exp\!\bigl(\mathbf{q}^\top \operatorname{RMSNorm}(\mathbf{k})\bigr)$.

**Queries and Keys:**

$$\mathbf{q}_l = \mathbf{w}_l \in \mathbb{R}^d \quad \text{(learnable per-layer vector)}, \qquad \mathbf{k}_i = \begin{cases} \mathbf{h}_1 & i = 0 \\ f_i(\mathbf{h}_i) & 1 \le i \le l{-}1 \end{cases}$$

The RMSNorm inside $\phi$ prevents layers with large-magnitude outputs from dominating. Complexity: $O(L^2 d)$ arithmetic, $O(Ld)$ memory — modest since $L \ll T$.

In [7]:
class FullAttnResLayer(nn.Module):
    """
    A single Transformer layer using Full Attention Residuals.

    Each sub-layer (attention, MLP) computes its input by attending over
    *all* previous layer outputs via a learned pseudo-query w_l rather than
    simply summing (standard residual).

    The attention weight from source i to layer l is:
        α_{i→l} = softmax_j( q_l^T · RMSNorm(k_j) )
    where q_l = w_l (learnable), k_i = v_i = layer output i.
    """

    def __init__(self, dim: int, n_heads: int, mlp_hidden: int | None = None):
        super().__init__()
        # sub-layers
        self.attn = CausalSelfAttention(dim, n_heads)
        self.mlp = FeedForward(dim, mlp_hidden)
        self.attn_norm = RMSNorm(dim)
        self.mlp_norm = RMSNorm(dim)

        # per-sub-layer pseudo-queries and key norms for depth attention
        self.attn_query = nn.Parameter(torch.randn(dim) * 0.02)
        self.mlp_query = nn.Parameter(torch.randn(dim) * 0.02)
        self.attn_key_norm = RMSNorm(dim)
        self.mlp_key_norm = RMSNorm(dim)

    @staticmethod
    def _depth_attention(query: Tensor, values: list[Tensor], key_norm: RMSNorm) -> Tensor:
        """
        Compute softmax attention over depth dimension.

        Args:
            query:   [D]          — learnable pseudo-query w_l
            values:  list of N tensors each [B, T, D] — preceding layer outputs
            key_norm: RMSNorm to normalize keys

        Returns:
            [B, T, D] — weighted combination of values
        """
        V = torch.stack(values, dim=0)          # [N, B, T, D]
        K = key_norm(V)                          # [N, B, T, D]
        # logits: q^T · RMSNorm(k_i) for each source i
        logits = torch.einsum("d, n b t d -> n b t", query, K)  # [N, B, T]
        weights = logits.softmax(dim=0)                          # [N, B, T]
        h = torch.einsum("n b t, n b t d -> b t d", weights, V) # [B, T, D]
        return h

    def forward(
        self, layer_outputs: list[Tensor]
    ) -> tuple[list[Tensor], Tensor]:
        """
        Args:
            layer_outputs: list of all preceding layer outputs
                           (starts with token embeddings h_1)

        Returns:
            updated layer_outputs list (with two new entries: attn_out, mlp_out)
        """
        # --- attention sub-layer ---
        h = self._depth_attention(self.attn_query, layer_outputs, self.attn_key_norm)
        attn_out = self.attn(self.attn_norm(h))
        layer_outputs.append(attn_out)

        # --- MLP sub-layer ---
        h = self._depth_attention(self.mlp_query, layer_outputs, self.mlp_key_norm)
        mlp_out = self.mlp(self.mlp_norm(h))
        layer_outputs.append(mlp_out)

        return layer_outputs


class FullAttnResTransformer(nn.Module):
    """
    Transformer with Full Attention Residuals.

    Each layer attends over all previous layer outputs (including the
    token embedding) instead of consuming a single accumulated residual.
    """

    def __init__(
        self,
        vocab_size: int,
        dim: int,
        n_layers: int,
        n_heads: int,
        max_seq_len: int = 2048,
    ):
        super().__init__()
        self.tok_emb = nn.Embedding(vocab_size, dim)
        self.pos_emb = nn.Embedding(max_seq_len, dim)

        self.layers = nn.ModuleList(
            [FullAttnResLayer(dim, n_heads) for _ in range(n_layers)]
        )

        # final aggregation: attend over all layer outputs to produce logits
        self.final_query = nn.Parameter(torch.randn(dim) * 0.02)
        self.final_key_norm = RMSNorm(dim)
        self.out_norm = RMSNorm(dim)
        self.lm_head = nn.Linear(dim, vocab_size, bias=False)

    def forward(self, idx: Tensor) -> Tensor:
        """
        Args:
            idx: [B, T] token indices
        Returns:
            logits: [B, T, vocab_size]
        """
        B, T = idx.shape
        positions = torch.arange(T, device=idx.device).unsqueeze(0)
        h1 = self.tok_emb(idx) + self.pos_emb(positions)  # [B, T, D]

        layer_outputs: list[Tensor] = [h1]  # b_0 = h_1 (token embeddings)

        for layer in self.layers:
            layer_outputs = layer(layer_outputs)

        # final output aggregation via depth attention
        h = FullAttnResLayer._depth_attention(
            self.final_query, layer_outputs, self.final_key_norm
        )
        return self.lm_head(self.out_norm(h))


print("Full Attention Residuals defined")

Full Attention Residuals defined


## 2. Block Attention Residuals

Full AttnRes stores all $L$ layer outputs — $O(Ld)$ memory. **Block AttnRes** partitions layers into $N$ blocks and attends over **block-level** representations, reducing overhead to $O(Nd)$.

### Intra-Block Accumulation
Layers within a block are summed via standard residuals:

$$\mathbf{b}_n = \sum_{j \in \mathcal{B}_n} f_j(\mathbf{h}_j)$$

$\mathbf{b}_n^i$ denotes the partial sum over the first $i$ layers in block $n$.

### Inter-Block Attention
For the $i$-th layer in block $n$, the value matrix attending over is:

$$\mathbf{V} = \begin{cases} [\mathbf{b}_0,\, \mathbf{b}_1,\, \ldots,\, \mathbf{b}_{n-1}]^\top & i = 1 \;\text{(first layer of block)} \\ [\mathbf{b}_0,\, \mathbf{b}_1,\, \ldots,\, \mathbf{b}_{n-1},\, \mathbf{b}_n^{i-1}]^\top & i \ge 2 \;\text{(subsequent layers)} \end{cases}$$

Keys follow the same structure with RMSNorm, and attention weights are softmax over the depth dimension as before. $\mathbf{b}_0 = \mathbf{h}_1$ (token embeddings).

**Efficiency tradeoff:** $N = L$ recovers Full AttnRes; $N = 1$ reduces to standard residuals. Empirically $N \approx 8$ recovers most of the benefit.

In [8]:
def block_attn_res(
    blocks: list[Tensor],
    partial_block: Tensor,
    query: Tensor,
    norm: RMSNorm,
) -> Tensor:
    """
    Inter-block attention: attend over completed block representations
    plus the current intra-block partial sum.

    Args:
        blocks:        list of N tensors [B, T, D] — completed block reps
        partial_block: [B, T, D] — current intra-block partial sum (b_n^i)
        query:         [D] — learnable pseudo-query w_l
        norm:          RMSNorm for key normalization

    Returns:
        [B, T, D] — weighted combination
    """
    V = torch.stack(blocks + [partial_block], dim=0)   # [N+1, B, T, D]
    K = norm(V)                                         # [N+1, B, T, D]
    logits = torch.einsum("d, n b t d -> n b t", query, K)
    weights = logits.softmax(dim=0)                     # [N+1, B, T]
    h = torch.einsum("n b t, n b t d -> b t d", weights, V)
    return h


class BlockAttnResLayer(nn.Module):
    """
    Single Transformer layer with Block Attention Residuals.

    Maintains:
      - `blocks`: list of completed block representations [b_0, ..., b_{n-1}]
      - `partial_block`: intra-block running sum b_n^i

    At block boundaries the partial sum is pushed to `blocks` and reset.
    """

    def __init__(
        self,
        dim: int,
        n_heads: int,
        layer_idx: int,
        block_size: int,
        mlp_hidden: int | None = None,
    ):
        super().__init__()
        self.layer_idx = layer_idx
        # block_size counts sub-layers (attn + MLP); each transformer layer = 2
        self.block_size = block_size

        self.attn = CausalSelfAttention(dim, n_heads)
        self.mlp = FeedForward(dim, mlp_hidden)
        self.attn_norm = RMSNorm(dim)
        self.mlp_norm = RMSNorm(dim)

        # per-sub-layer pseudo-queries & key norms for inter-block attention
        self.attn_res_query = nn.Parameter(torch.randn(dim) * 0.02)
        self.mlp_res_query = nn.Parameter(torch.randn(dim) * 0.02)
        self.attn_res_norm = RMSNorm(dim)
        self.mlp_res_norm = RMSNorm(dim)

    def forward(
        self, blocks: list[Tensor], partial_block: Tensor
    ) -> tuple[list[Tensor], Tensor]:
        """
        Args:
            blocks:        completed block representations (starts with [h_1])
            partial_block: intra-block partial sum [B, T, D]
        Returns:
            (updated blocks, updated partial_block)
        """
        # --- inter-block attention before self-attention ---
        h = block_attn_res(blocks, partial_block, self.attn_res_query, self.attn_res_norm)

        # check block boundary (layer_idx is 0-based; boundary every block_size//2 layers)
        layers_per_block = self.block_size // 2
        if layers_per_block > 0 and self.layer_idx % layers_per_block == 0 and self.layer_idx > 0:
            blocks = blocks + [partial_block]  # push completed block
            partial_block = torch.zeros_like(partial_block)

        # --- self-attention sub-layer ---
        attn_out = self.attn(self.attn_norm(h))
        partial_block = partial_block + attn_out

        # --- inter-block attention before MLP ---
        h = block_attn_res(blocks, partial_block, self.mlp_res_query, self.mlp_res_norm)

        # --- MLP sub-layer ---
        mlp_out = self.mlp(self.mlp_norm(h))
        partial_block = partial_block + mlp_out

        return blocks, partial_block


class BlockAttnResTransformer(nn.Module):
    """
    Transformer with Block Attention Residuals.

    Layers are partitioned into N blocks. Within each block, layer outputs
    are accumulated via standard residuals. Across blocks, each layer
    attends over block-level representations via softmax depth attention.
    """

    def __init__(
        self,
        vocab_size: int,
        dim: int,
        n_layers: int,
        n_heads: int,
        n_blocks: int = 8,
        max_seq_len: int = 2048,
    ):
        super().__init__()
        self.tok_emb = nn.Embedding(vocab_size, dim)
        self.pos_emb = nn.Embedding(max_seq_len, dim)

        # block_size = number of sub-layers per block (attn + MLP each count as 1)
        block_size = (n_layers * 2) // n_blocks
        block_size = max(block_size, 2)  # at least one transformer layer per block

        self.layers = nn.ModuleList(
            [
                BlockAttnResLayer(dim, n_heads, layer_idx=i, block_size=block_size)
                for i in range(n_layers)
            ]
        )

        # final aggregation
        self.final_query = nn.Parameter(torch.randn(dim) * 0.02)
        self.final_norm = RMSNorm(dim)
        self.out_norm = RMSNorm(dim)
        self.lm_head = nn.Linear(dim, vocab_size, bias=False)

    def forward(self, idx: Tensor) -> Tensor:
        """
        Args:
            idx: [B, T] token indices
        Returns:
            logits: [B, T, vocab_size]
        """
        B, T = idx.shape
        positions = torch.arange(T, device=idx.device).unsqueeze(0)
        h1 = self.tok_emb(idx) + self.pos_emb(positions)

        blocks: list[Tensor] = [h1]      # b_0 = token embeddings
        partial_block = torch.zeros_like(h1)

        for layer in self.layers:
            blocks, partial_block = layer(blocks, partial_block)

        # final output: attend over all blocks + remaining partial sum
        all_reps = blocks + [partial_block]
        V = torch.stack(all_reps, dim=0)
        K = self.final_norm(V)
        logits_depth = torch.einsum("d, n b t d -> n b t", self.final_query, K)
        weights = logits_depth.softmax(dim=0)
        h = torch.einsum("n b t, n b t d -> b t d", weights, V)

        return self.lm_head(self.out_norm(h))


print("Block Attention Residuals defined")

Block Attention Residuals defined


## 3. Forward Pass Comparison

Instantiate both models with the same hyperparameters and verify they produce valid logits on random input. We also compare parameter counts to illustrate the minimal overhead of AttnRes.

In [5]:
# ── Hyperparameters ──
VOCAB   = 256
DIM     = 128
LAYERS  = 8
HEADS   = 4
BLOCKS  = 4       # for Block AttnRes
SEQ_LEN = 32
BATCH   = 2

device = "cuda" if torch.cuda.is_available() else "cpu"

# ── Instantiate models ──
full_model  = FullAttnResTransformer(VOCAB, DIM, LAYERS, HEADS, max_seq_len=SEQ_LEN).to(device)
block_model = BlockAttnResTransformer(VOCAB, DIM, LAYERS, HEADS, n_blocks=BLOCKS, max_seq_len=SEQ_LEN).to(device)

# ── Random input ──
x = torch.randint(0, VOCAB, (BATCH, SEQ_LEN), device=device)

# ── Forward passes ──
with torch.no_grad():
    logits_full  = full_model(x)
    logits_block = block_model(x)

print(f"Input shape:              {list(x.shape)}")
print(f"Full AttnRes logits:      {list(logits_full.shape)}")
print(f"Block AttnRes logits:     {list(logits_block.shape)}")
print()

def count_params(m: nn.Module) -> int:
    return sum(p.numel() for p in m.parameters())

print(f"Full AttnRes  params:     {count_params(full_model):,}")
print(f"Block AttnRes params:     {count_params(block_model):,}")
print(f"Overhead (Block vs Full): {count_params(block_model) - count_params(full_model):+,}")

Input shape:              [2, 32]
Full AttnRes logits:      [2, 32, 256]
Block AttnRes logits:     [2, 32, 256]

Full AttnRes  params:     2,173,312
Block AttnRes params:     2,173,312
Overhead (Block vs Full): +0
